# 32 — Resume Section Detection
**Goal:** Identify section boundaries in unstructured resume text.

## 1. Common Resume Sections

In [ ]:
SECTIONS = [
    "summary", "objective", "profile",
    "experience", "work history", "employment",
    "education", "academic",
    "skills", "technical skills", "core competencies",
    "projects", "certifications", "publications",
    "languages", "interests", "references",
]
print(f"Known sections: {len(SECTIONS)}")
for s in SECTIONS: print(f"  - {s}")

## 2. Regex-Based Section Detection

In [ ]:
import re

SECTION_PATTERNS = {s: re.compile(r"^" + re.escape(s), re.IGNORECASE) for s in SECTIONS}

def detect_sections(text):
    lines = text.split("\n")
    sections = []
    for i, line in enumerate(lines):
        line_stripped = line.strip()
        if not line_stripped: continue
        for section_name, pattern in SECTION_PATTERNS.items():
            if pattern.search(line_stripped):
                # Check it looks like a header (short, possibly all-caps)
                if len(line_stripped) < 40:
                    sections.append((section_name, i, line_stripped))
                    break
    return sections

resume = """SUMMARY
Data scientist with Python and ML experience.

EXPERIENCE
Google — Senior Data Scientist, 2020-Present
Built ML pipelines.

EDUCATION
M.S. Computer Science, Stanford University

SKILLS
Python, TensorFlow, PyTorch, SQL, AWS
"""
for name, idx, header in detect_sections(resume):
    print(f"  Line {idx}: [{name:15s}] '{header}'")

## 3. ML-Based Section Classification

In [ ]:
# Use zero-shot classification for section detection
from transformers import pipeline
try:
    classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
    lines = ["Professional Summary", "Work Experience", "Education", "Skills & Expertise"]
    for line in lines:
        result = classifier(line, SECTIONS)
        print(f"  '{line:25s}' -> {result['labels'][0]:15s} ({result['scores'][0]:.2f})")
except:
    print("Transformers not available. Regex approach works fine for most resumes.")

## 4. Section Content Extraction

In [ ]:
def extract_section_content(text, section_name):
    """Extract all content under a detected section."""
    lines = text.split("\n")
    in_section = False
    content = []
    for line in lines:
        ls = line.strip()
        if not ls: continue
        # Check if this line is a section header
        is_header = False
        for sn, pat in SECTION_PATTERNS.items():
            if pat.search(ls) and len(ls) < 40:
                if in_section:
                    in_section = False  # found next section
                if sn == section_name.lower():
                    in_section = True
                break
        else:
            if in_section and ls:
                content.append(ls)
    return content

print("Content under 'experience':")
print(extract_section_content(resume, "experience"))

## Summary: Regex + heuristics for basic detection. Zero-shot ML for complex layouts.